## 1. Import Libraries

- **pandas** : load and handle the dataset
- **train_test_split** : split data into training and testing sets
- **CountVectorizer**:convert text messages into word counts
- **MultinomialNB**: Naive Bayes model for text classification
- **accuracy_score / classification_report** :evaluate the model

In [101]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

## 2. Load the Dataset

- **`encoding='latin-1'`**: I needed that because the file contains special characters that default encoding (UTF-8) can't read properly.

The dataset is now stored in `df`.

In [102]:
df = pd.read_csv(r"C:\Users\Junayed\spam-detection-model\spam_data.csv", encoding='latin-1')

In [103]:
df.head(5)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


## 3. Keep Only Needed Columns

- **`v1`**: the label (`ham` or `spam`)
- **`v2`**: the message text

`.head(5)` shows the first 5 rows to confirm it looks right.

In [104]:
df = df[['v1', 'v2']]
df.head(5)

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## 4. Rename Columns

The original column names `v1` and `v2` aren't clear, so need to  rename them:

- **`v1` → `label`**: spam or ham
- **`v2` → `message`**: the SMS text

Now the columns are easy to understand.

In [105]:
df.columns = ['label', 'message']

In [106]:
df.head(5)

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## 5. Check Class Distribution

- **ham:** 4825 (≈87%)
- **spam:** 747 (≈13%)

The data is **imbalanced**; most messages are ham. This means accuracy alone can be misleading (a model that always predicts "ham" would still score ~87%). We'll need to look at **precision and recall** for spam later.

In [107]:
df["label"].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [108]:
print("Before:", len(df))
df = df.drop_duplicates(subset="message").reset_index(drop=True)
print("After:", len(df))

Before: 5572
After: 5169


## 6. Convert Labels to Numbers

ML models can't work with text labels, so need to  convert them to numbers:

- **spam → 1**
- **ham → 0**

We do this with a small function applied to every row, and store the result in a new column **`label_dif`**.

The output confirms it worked, spam rows now show `1`, ham rows show `0`.

In [109]:
def label_diff(label):
    if label == "spam":
        return 1
    else:
        return 0

df["label_dif"] = df["label"].apply(label_diff)
df.head(5)

,label,message,label_dif
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


## 7. Convert Text to Numbers

Models can't read text, so we turn each message into a vector of word counts using **`CountVectorizer`**.

- **`stop_words='english'`**: removes common words like "the", "is", "and" that carry no meaning
- **`max_features=5000`**: keeps only the 5000 most frequent words (reduces noise and keeps the model fast)

**Result:**
- **`X`**: the messages as a matrix of word counts (features)
- **`Y`**: the target labels (0 = ham, 1 = spam)

In [110]:
cv = CountVectorizer(stop_words='english', max_features=5000)

X = cv.fit_transform(df['message'])
Y = df['label_dif']

## 8. View English Stopwords (NLTK)

We download and print NLTK's list of **English stopwords** common words that add little meaning.

- These are the same kind of words `CountVectorizer(stop_words='english')` removes.
- Seeing them helps us understand **what gets dropped** before the model sees the text.


In [111]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Junayed\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 9. Build a Noise List

combine two sources of "noise" into one list:

- **Stopwords**: common English words (e.g. "the", "is", "and")
- **Punctuation**: characters like `!`, `?`, `.`, `,`

This `noise` list will be used to clean the messages before feeding them to the model.

In [112]:
import string
noise = stopwords.words('english') + list(string.punctuation)
print(noise)

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

## 10. Split Data into Train and Test

- **Training set (80%)**: the model learns from this
- **Test set (20%)**: we use this to check how well the model performs on unseen data

- **`random_state=43`**: makes the split reproducible (same split every run)

In [113]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size=0.2, random_state= 43)

## 11. Train the Model & Evaluate

Now train a **Multinomial Naive Bayes** model on the training data, then predict on the test set.

### Results

- **Accuracy: 98.39%**: very high, but let's look deeper.

| Class | Precision | Recall | F1-score | Support |
|---|---|---|---|---|
| Ham (not spam) | 0.99 | 0.99 | 0.99 | 960 |
| Spam | 0.94 | 0.95 | 0.94 | 155 |

### What this means

- **Precision (spam) = 0.94** → When the model says "spam", it's right 94% of the time.
- **Recall (spam) = 0.95** → It catches 95% of actual spam messages.
- **F1 (spam) = 0.94** → Strong balance of both.

### Why this matters (imbalanced data)

Even though 87% of messages are ham, the model still performs well on the **minority class (spam)**. This proves it learned real spam patterns not just "predict ham to look accurate."

**Conclusion:** The model reliably detects spam while rarely flagging real messages.

In [114]:
model = MultinomialNB()
model.fit(X_train, Y_train)
Y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(Y_test, Y_pred):.2%}")
print(classification_report(Y_test, Y_pred,
target_names=["Ham (not spam)", 'spam']))

Accuracy: 98.07%
                precision    recall  f1-score   support

Ham (not spam)       0.99      0.99      0.99       895
          spam       0.92      0.94      0.93       139

      accuracy                           0.98      1034
     macro avg       0.95      0.96      0.96      1034
  weighted avg       0.98      0.98      0.98      1034



## 12. Test the Model on New Messages

built a small helper function `check_spam()` that:

1. **Takes a new message** as input
2. **Converts it** into word counts using the same `CountVectorizer` (`cv.transform`)
3. **Predicts** the label (0 = ham, 1 = spam)
4. **Prints the probability** so we see how confident the model is

### Example

**Input:** `"Congratulations! You won a free iPhone. Click now!"`  
**Output:** `Spam (Probability: 99.77%)`

In [115]:
def check_spam(message):
    msg_vec = cv.transform([message])
    pred = model.predict(msg_vec)[0]
    prob = model.predict_proba(msg_vec)[0]
    label= 'Spam' if pred == 1 else "Not spam"
    print(f"Message: {message}")
    print(f"Prediction: {label} (Probability: {prob[pred]:.2%})")

check_spam("Congratulations! You won a free iPhone. Click now!")

Message: Congratulations! You won a free iPhone. Click now!
Prediction: Spam (Probability: 99.77%)


In [116]:
check_spam('Hey, are we still meeting tomorrow at 3pm?')
check_spam('URGENT: Your account has been compromised. Call us!')


Message: Hey, are we still meeting tomorrow at 3pm?
Prediction: Not spam (Probability: 99.97%)
Message: URGENT: Your account has been compromised. Call us!
Prediction: Spam (Probability: 83.75%)


In [117]:
check_spam("As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Callertune")

check_spam("Todays Voda numbers ending 7548 are selected to receive a $350 award. If you have a match please call 08712300220 quoting claim code 4041 standard rates app")

Message: As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Callertune
Prediction: Not spam (Probability: 99.98%)
Message: Todays Voda numbers ending 7548 are selected to receive a $350 award. If you have a match please call 08712300220 quoting claim code 4041 standard rates app
Prediction: Spam (Probability: 100.00%)
